# v31 — v30 recipe + ResNet-50 backbone (teacher's L1a slide 30 primary recommendation)

**This is v30 with `BACKBONE = "resnet50"`. Single change. Everything else identical to v30.**

## Why this exists

The teacher's L1a slide 30 (Lu et al. 2020) explicitly recommends **"ResNet-50 or DenseNet-201"**
for cell-level cancer classification. We've been avoiding ResNet-50 since v21 (LB 0.7018) but
v21 was confounded: it changed the backbone AND added a 224 bilinear upscale (against Lu 2020's
"no interpolation — texture matters" warning).

v31 finally tests ResNet-50 cleanly:
- **128×128 native input** (no upscale, respects Lu 2020)
- **Full 12 patients** in training (v19 / v30 regime — no broken val holdout)
- **Discriminative LR** (head 3e-4, backbone 3e-5, per L4 slide 96)
- All other v19/v30 components intact (MIL, strong aug, AdaBN, 8-way D4 TTA, stain norm)

## Single change vs v30

| Lever | v30 (EffNet-B0) | **v31** (ResNet-50) |
|---|---|---|
| Backbone | EfficientNet-B0 | **ResNet-50** |
| Params per branch | ~4.0M | ~23.5M |
| Total params | ~9.3M | ~52M |
| Feature dim | 1280 | 2048 |
| Everything else | (v30) | **same as v30** |

## Single change vs v21 (which failed at 0.7018)

| Lever | v21 (LB 0.7018) | **v31** |
|---|---|---|
| Backbone | ResNet-50 | ResNet-50 ✓ |
| Input | 224 bilinear upscale | **128 native (no interp)** |
| Optimizer | flat LR | **disc LR** |
| Other | v19 stack | v19 stack ✓ |

## Compute on T4

ResNet-50 at 128 native is cheap (FLOPs scale as `(128/224)² ≈ 0.33` of ResNet-50 @ 224):

| Stage | Time |
|---|---|
| JPEG cache | ~56 min |
| Pixel stats | ~30s |
| Train 12 epochs (~5 min/ep for ResNet-50 @ 128) | ~60 min |
| 8-way D4 TTA + AdaBN | ~8 min |
| **Total** | **~2h 5min** |

Batch 128 fits comfortably on T4 (v27 used only 26% of VRAM with EffNet-B0; ResNet-50 will use ~50%).

## Reading the v31 LB (vs v30's LB)

| v31 vs v30 | Verdict |
|---|---|
| **v31 > v30 + 0.01** | ResNet-50 capacity helps. Teacher's primary recommendation validated. Try v32 too. |
| **v31 ≈ v30** | Both backbones converge with disc LR. EffNet-B0 is the cheaper choice. |
| **v31 < v30 - 0.01** | ResNet-50 still overfits more, even at native resolution. EffNet-B0 wins. |
| **v31 ≥ 0.76** | We're competitive with the LB leader (0.7832). |

## IMPORTANT — Commit ONLY after v30 lands

If v30 < 0.745: disc LR didn't help on EffNet-B0. It probably won't help on ResNet-50 either.
Skip v31, debug instead.

If v30 ≥ 0.745: disc LR helped. Commit v31 (and v32) to test backbone diversity.

To commit:
1. **Save Version** → **Save & Run All (Commit)**
2. Description: `v31: v30 recipe + ResNet-50 @ 128 native`
3. **Wait ~2h 5min**
4. Output tab → submission.csv

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === v30 BACKBONE selection — change this single line for v31/v32 ===
# One of: "efficientnet_b0" (v19/v30 default), "resnet50", "densenet201", "resnet18".
BACKBONE = "resnet50"

# === v30 KEY CHANGE vs v19: discriminative LR (L4 slide 96) ===
BACKBONE_LR_RATIO = 0.1   # backbone gets LR * 0.1; head gets LR

# === Inherited from v19 (unchanged) ===
USE_MIL_LOSS        = True
MIL_WEIGHT          = 0.5
USE_STRONG_AUG      = True
RANDOM_ERASING_P    = 0.25

USE_TEST_STAIN_NORM = True
USE_ADABN           = True
USE_MULTISCALE_TTA  = False
TTA_SCALES          = (112, 128, 144)

LABEL_SMOOTHING     = 0.0

# DenseNet safety (gradient checkpointing in dense blocks).
DENSENET_MEMORY_EFFICIENT = True

# === Training (matches v19 exactly except disc LR) ===
BASE_SEED   = 1     # same seed as v19 — direct comparability
EPOCHS      = 12    # v19's schedule (NOT v27's 15)
BATCH_SIZE  = 128
LR          = 3e-4  # head LR (backbone gets LR * BACKBONE_LR_RATIO = 3e-5)
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.0
DROPOUT     = 0.3

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

_VALID_BACKBONES = {"efficientnet_b0", "resnet50", "densenet201", "resnet18"}
assert BACKBONE in _VALID_BACKBONES, \
    f"Unknown BACKBONE={BACKBONE!r}, must be one of {_VALID_BACKBONES}"

print(f"\nConfig (v31 — v30 recipe + resnet50 backbone):")
print(f"  BACKBONE             = {BACKBONE}")
print(f"  BACKBONE_LR_RATIO    = {BACKBONE_LR_RATIO}  (head LR={LR}, backbone LR={LR*BACKBONE_LR_RATIO})")
print(f"  USE_MIL_LOSS         = {USE_MIL_LOSS}  weight={MIL_WEIGHT}")
print(f"  USE_STRONG_AUG       = {USE_STRONG_AUG}  erasing_p={RANDOM_ERASING_P}")
print(f"  USE_TEST_STAIN_NORM  = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN            = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA   = {USE_MULTISCALE_TTA}  scales={TTA_SCALES}")
print(f"  EPOCHS               = {EPOCHS}  BATCH_SIZE = {BATCH_SIZE}  BASE_SEED = {BASE_SEED}")
print(f"  MIXUP_ALPHA          = {MIXUP_ALPHA}  (disabled)")
print(f"  Val holdout          = NONE — full 12-patient training (v19 regime)")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
# === Backbone builders — one per supported architecture, dispatched by BACKBONE string ===

def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features  # 2048
    net.fc = nn.Identity()
    return net, fd

def _make_densenet201_branch(pretrained=True):
    """torchvision densenet201 with 1-ch stem + memory_efficient gradient checkpointing."""
    weights = "DEFAULT" if pretrained else None
    net = models.densenet201(weights=weights, memory_efficient=DENSENET_MEMORY_EFFICIENT)
    old = net.features.conv0
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.features.conv0 = new_conv
    fd = net.classifier.in_features  # 1920
    net.classifier = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

_BACKBONE_FACTORY = {
    "resnet18":        _make_resnet18_branch,
    "resnet50":        _make_resnet50_branch,
    "densenet201":     _make_densenet201_branch,
    "efficientnet_b0": _make_effnet_b0_branch,
}

def make_branch(backbone, pretrained=True):
    if backbone not in _BACKBONE_FACTORY:
        raise ValueError(f"Unknown backbone {backbone!r}")
    return _BACKBONE_FACTORY[backbone](pretrained)


class MultimodalClassifier(nn.Module):
    """Dual-branch (BF + FL) classifier with late concat fusion. `backbone` selects arch."""
    def __init__(self, backbone, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.backbone_name = backbone
        self.bf_branch, fd = make_branch(backbone, pretrained)
        self.fl_branch, _  = make_branch(backbone, pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(backbone=BACKBONE, pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Output shape: {_m(_x, _x).shape}   params: {n_params / 1e6:.1f}M")
    print(f"Backbone: {BACKBONE}")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + small rotation, applied identically to BF and FL (paired)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        # v19 NEW: paired affine — same translate applied to both modalities so
        # BF/FL stay registered.
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        return bf, fl

def train_modality_transform(modality):
    """v19: stronger color jitter, with optional RandomErasing applied after normalization."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
        if RANDOM_ERASING_P > 0:
            # RandomErasing operates on normalized tensors; value=0 means it erases to the
            # normalized 0 (which corresponds to original-pixel = mean).
            steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                         ratio=(0.3, 3.3), value=0.0))
        return T.Compose(steps)
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    """v19 paired aug: D4 + ±10° rot, plus ±15° affine and 10% translate when strong aug is on."""
    if USE_STRONG_AUG:
        return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10)
    return PairedGeoAug(max_rot=10.0)

print(f"Augmentation summary:")
print(f"  ColorJitter:      {'brightness/contrast=0.4' if USE_STRONG_AUG else 'brightness/contrast=0.2'}")
print(f"  RandomErasing:    p={RANDOM_ERASING_P if USE_STRONG_AUG else 0.0}")
print(f"  PairedGeoAug:     D4 + ±10° rot" + (" + ±15° affine + 10% translate" if USE_STRONG_AUG else ""))

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """Per-patient mean-logit BCE loss (averages cell-logits within each patient)."""
    unique_pids = torch.unique(patient_ids)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits, p_labels = [], []
    for pid in unique_pids:
        mask = patient_ids == pid
        p_logits.append(logits[mask].mean())
        p_labels.append(y[mask][0])
    p_logits = torch.stack(p_logits); p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, log_every=200):
    model.train()
    losses, hard_ys, ps, pids_all = [], [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        pids_all.append(batch["patient_id"].numpy())
        y_s = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss_cell = criterion_cell(logits, y_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            old_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale: sched.step()
        else:
            loss.backward()
            if GRAD_CLIP > 0: nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); sched.step()
        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f} "
                  f"mil {float(np.mean(mil_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps); pids_all = np.concatenate(pids_all)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    # Per-patient mean prob (training-set diagnostic — shows which patients model fits/misses).
    per_pat = {}
    for pid in np.unique(pids_all):
        mask = pids_all == pid
        per_pat[int(pid)] = {
            "label": int(hard_ys[mask][0]),
            "mean_prob": float(ps[mask].mean()),
            "n_cells_seen": int(mask.sum()),
        }
    return (float(np.mean(losses)), float(np.mean(cell_losses)),
            float(np.mean(mil_losses)), auc, per_pat)


# === v30: NO val split — full 12-patient training (v19 regime) ===
all_pids = sorted(df_train["patient_id"].unique())
pat_label_by_pid = df_train.groupby("patient_id")["Diagnosis"].first().to_dict()
print(f"\nFull-data training (v19 regime):")
print(f"  All {len(all_pids)} patients used for training: {all_pids}")
print(f"  Labels: {pat_label_by_pid}")
print(f"  Total cells: {len(df_train)}  pos_rate: {df_train['Diagnosis'].mean():.4f}")


# === Diagnostic 7: sample BF/FL image preview (kept from v27; uses own RNG) ===
try:
    preview_rng = np.random.default_rng(42)
    train_cancer  = [p for p in all_pids if pat_label_by_pid[p] == 1]
    train_healthy = [p for p in all_pids if pat_label_by_pid[p] == 0]

    rows = []
    # Sample 2 cancer and 2 healthy training patients.
    for pid in preview_rng.choice(train_cancer,  size=min(2, len(train_cancer)),  replace=False):
        rows.append((int(pid), "train", 1))
    for pid in preview_rng.choice(train_healthy, size=min(2, len(train_healthy)), replace=False):
        rows.append((int(pid), "train", 0))

    n_per_pat = 2
    fig, axes = plt.subplots(len(rows), n_per_pat * 2,
                             figsize=(n_per_pat * 4, len(rows) * 2.2))
    if len(rows) == 1:
        axes = axes[np.newaxis, :]
    for row, (pid, split, lbl) in enumerate(rows):
        sub = df_train[df_train["patient_id"] == pid]
        sel_idxs = preview_rng.choice(sub.index.values,
                                       size=min(n_per_pat, len(sub)),
                                       replace=False)
        lbl_str = "CANCER" if lbl == 1 else "HEALTHY"
        for col, idx in enumerate(sel_idxs):
            name = df_train.loc[idx, "Name"]
            bf_img = np.asarray(Image.open(io.BytesIO(bf_train_cache[name])).convert("L"))
            fl_img = np.asarray(Image.open(io.BytesIO(fl_train_cache[name])).convert("L"))
            axes[row, col*2  ].imshow(bf_img, cmap="gray", vmin=0, vmax=255)
            axes[row, col*2  ].axis("off")
            axes[row, col*2  ].set_title(f"pat_{pid} ({split} {lbl_str})\nBF", fontsize=8)
            axes[row, col*2+1].imshow(fl_img, cmap="gray", vmin=0, vmax=255)
            axes[row, col*2+1].axis("off")
            axes[row, col*2+1].set_title("FL", fontsize=8)
    plt.suptitle("Sample BF/FL cells (all 12 patients used in training — no val holdout in v30)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_images.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  Saved sample_images.png ({len(rows)} rows)")
except Exception as _e:
    print(f"  [warn] image preview skipped: {_e}")


# === Build datasets and loaders (full data — no val split) ===
print(f"\n=== v31: Training (backbone={BACKBONE}, all {len(all_pids)} patients, {EPOCHS} epochs) ===")
seed_everything(BASE_SEED + 100)
train_ds = CachedCellDataset(df_train, bf_train_cache, fl_train_cache,
                             train_modality_transform("bf"),
                             train_modality_transform("fl"),
                             paired_tf=build_paired_aug())
sampler = PatientBalancedSampler(df_train, batch_size=BATCH_SIZE,
                                 patients_per_batch=PATIENTS_PER_BATCH, seed=BASE_SEED + 100)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

# === Build model with discriminative LR (the one change vs v19) ===
model = MultimodalClassifier(backbone=BACKBONE, pretrained=True, dropout=DROPOUT).to(DEVICE)
pos = (df_train["Diagnosis"] == 1).sum()
neg = (df_train["Diagnosis"] == 0).sum()
pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)

backbone_params = list(model.bf_branch.parameters()) + list(model.fl_branch.parameters())
head_params = list(model.head.parameters())
n_back = sum(p.numel() for p in backbone_params)
n_head = sum(p.numel() for p in head_params)
backbone_lr = LR * BACKBONE_LR_RATIO

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": backbone_lr},
    {"params": head_params,     "lr": LR},
], weight_decay=WEIGHT_DECAY)

print(f"  pos_weight={pos_weight.item():.3f}")
print(f"  head_LR={LR}  backbone_LR={backbone_lr}  (ratio = {BACKBONE_LR_RATIO})")
print(f"  backbone params: {n_back/1e6:.1f}M    head params: {n_head/1e6:.2f}M")
print(f"  MIL_W={MIL_WEIGHT if USE_MIL_LOSS else 0.0}  epochs={EPOCHS}  batch={BATCH_SIZE}")
print(f"  n_train_batches={len(train_loader)}")

criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[backbone_lr, LR],
    steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1,
)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None


# === Diagnostic 8: LR schedule preview (verify disc LR is wired correctly) ===
try:
    _preview_opt = torch.optim.AdamW([
        {"params": [torch.zeros(1, requires_grad=True)], "lr": backbone_lr},
        {"params": [torch.zeros(1, requires_grad=True)], "lr": LR},
    ], weight_decay=WEIGHT_DECAY)
    _preview_sch = torch.optim.lr_scheduler.OneCycleLR(
        _preview_opt, max_lr=[backbone_lr, LR],
        steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1,
    )
    _bb_lrs, _hd_lrs = [], []
    _n_steps = len(train_loader) * EPOCHS
    for _ in range(_n_steps):
        _bb_lrs.append(_preview_opt.param_groups[0]["lr"])
        _hd_lrs.append(_preview_opt.param_groups[1]["lr"])
        _preview_sch.step()

    fig, ax = plt.subplots(figsize=(10, 3))
    _x = np.arange(_n_steps)
    ax.plot(_x, _bb_lrs, label=f"backbone (max={backbone_lr:.1e})", color="tab:blue")
    ax.plot(_x, _hd_lrs, label=f"head (max={LR:.1e})",              color="tab:red")
    for _e in range(EPOCHS + 1):
        ax.axvline(_e * len(train_loader), ls=":", color="gray", alpha=0.3)
    ax.set_yscale("log")
    ax.legend(); ax.grid(True, alpha=0.4)
    ax.set(title=f"OneCycleLR schedule preview (pct_start=0.1, {EPOCHS} epochs × {len(train_loader)} steps)",
           xlabel="optimizer step (dotted = epoch boundary)", ylabel="LR (log)")
    plt.tight_layout()
    plt.savefig("/kaggle/working/lr_schedule.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  Saved lr_schedule.png  (peak backbone={max(_bb_lrs):.2e}, peak head={max(_hd_lrs):.2e})")
    del _preview_opt, _preview_sch, _bb_lrs, _hd_lrs
except Exception as _e:
    print(f"  [warn] LR schedule preview skipped: {_e}")


# === Train loop (v19 regime: save last, no val) ===
ckpt_path = OUT_DIR / "fulldata_last.pt"
history = []
gpu_peak_reported = False
for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_cell, tr_mil, tr_auc, tr_per_pat = run_epoch_train(
        model, train_loader, optimizer, scaler, criterion_cell, sched,
        pos_weight=pos_weight)
    dt = time.time() - t0

    # Compact per-patient summary: spread of mean prob within cancer/healthy groups.
    cancer_means  = [d["mean_prob"] for d in tr_per_pat.values() if d["label"] == 1]
    healthy_means = [d["mean_prob"] for d in tr_per_pat.values() if d["label"] == 0]
    if cancer_means and healthy_means:
        cm_mean = float(np.mean(cancer_means)); cm_std = float(np.std(cancer_means))
        hm_mean = float(np.mean(healthy_means)); hm_std = float(np.std(healthy_means))
        sep = cm_mean - hm_mean
        pat_summary = (f"C_pat:{cm_mean:.3f}±{cm_std:.3f}  "
                       f"H_pat:{hm_mean:.3f}±{hm_std:.3f}  sep:{sep:+.3f}")
    else:
        pat_summary = ""

    print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f}  cell {tr_cell:.4f}  mil {tr_mil:.4f}  "
          f"tr_auc {tr_auc:.4f}  | {pat_summary}  | {dt:.1f}s")

    history.append({
        "epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell, "tr_mil": tr_mil,
        "tr_auc": tr_auc, "per_patient_train": tr_per_pat, "time": dt,
    })

    # One-time GPU peak memory after epoch 0.
    if ep == 0 and torch.cuda.is_available() and not gpu_peak_reported:
        peak     = torch.cuda.max_memory_allocated() / (1024**3)
        reserved = torch.cuda.max_memory_reserved()  / (1024**3)
        total    = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"  [mem] peak GPU after ep 0: alloc={peak:.2f}GB  "
              f"reserved={reserved:.2f}GB  / total={total:.1f}GB ({peak/total:.0%})")
        torch.cuda.reset_peak_memory_stats()
        gpu_peak_reported = True

# === v19 regime: save LAST epoch ===
torch.save({
    "model": model.state_dict(),
    "epoch": EPOCHS - 1,
    "args": {"dropout": DROPOUT, "backbone": BACKBONE,
             "train_auc_last": history[-1]["tr_auc"]},
}, ckpt_path)
print(f"\n=== Training complete ===")
print(f"Final train AUC: {history[-1]['tr_auc']:.4f}")
print(f"Saved last-epoch ckpt: {ckpt_path}")

print(f"\nFinal per-patient train mean prob (after epoch {EPOCHS-1}):")
final_pp = history[-1]["per_patient_train"]
for pid in sorted(final_pp.keys()):
    d = final_pp[pid]
    lab = "CANCER " if d["label"] == 1 else "HEALTHY"
    print(f"  pat_{pid:>2d} ({lab}): mean_prob={d['mean_prob']:.4f}  n_cells_seen={d['n_cells_seen']}")

with open(OUT_DIR / "history.json", "w") as f:
    json.dump({"backbone": BACKBONE, "epochs": EPOCHS,
               "final_train_auc": history[-1]["tr_auc"],
               "history": history}, f, indent=2)

del model, optimizer, sched, scaler, train_loader, train_ds, sampler
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# v30 has no val data — show train metrics only (1x3 layout).
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = [e["epoch"] for e in history]

# (1) Train losses
ax = axes[0]
ax.plot(epochs, [e["tr_loss"] for e in history], marker="o", color="tab:blue",   label="total")
ax.plot(epochs, [e["tr_cell"] for e in history], marker="s", color="tab:purple", label="cell BCE")
ax.plot(epochs, [e["tr_mil"]  for e in history], marker="^", color="tab:orange", label="MIL")
ax.legend(); ax.grid(True)
ax.set(title="Train losses", xlabel="epoch", ylabel="loss")

# (2) Train AUC + per-patient mean prob trajectories
ax = axes[1]
ax.plot(epochs, [e["tr_auc"] for e in history], marker="o", color="tab:green",
        label="train AUC (cell)", linewidth=2)
# Per-patient trajectories (12 thin lines colored by class).
all_train_pids = sorted(history[0]["per_patient_train"].keys())
for pid in all_train_pids:
    label = history[0]["per_patient_train"][pid]["label"]
    color = "tab:red" if label == 1 else "tab:blue"
    trajectory = [h["per_patient_train"][pid]["mean_prob"] for h in history]
    ax.plot(epochs, trajectory, marker=".", color=color, alpha=0.4, linewidth=1)
# Legend handles for the patient lines
from matplotlib.lines import Line2D
patient_legend = [
    Line2D([0], [0], color="tab:red",   alpha=0.6, lw=1, label="cancer patient (mean prob)"),
    Line2D([0], [0], color="tab:blue",  alpha=0.6, lw=1, label="healthy patient (mean prob)"),
    Line2D([0], [0], color="tab:green", lw=2,             label="train AUC (cell)"),
]
ax.legend(handles=patient_legend, loc="best")
ax.axhline(0.5, ls=":", color="gray", alpha=0.5)
ax.grid(True)
ax.set(title="Train AUC + per-patient mean prob", xlabel="epoch", ylabel="AUC / mean P(cancer)")
ax.set_ylim(-0.05, 1.05)

# (3) Epoch time
ax = axes[2]
ax.plot(epochs, [e["time"] for e in history], marker="o", color="tab:red")
ax.grid(True)
ax.set(title="Epoch time (s)", xlabel="epoch", ylabel="seconds")

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Compact text summary table.
print(f"\nEpoch-by-epoch summary (train only — no val):")
hdr = f"  {'ep':>3}  {'tr_loss':>7}  {'tr_auc':>7}  {'C_pat_mean':>11}  {'H_pat_mean':>11}  {'sep':>7}"
print(hdr)
for e in history:
    pp = e["per_patient_train"]
    cm = np.mean([d["mean_prob"] for d in pp.values() if d["label"] == 1])
    hm = np.mean([d["mean_prob"] for d in pp.values() if d["label"] == 0])
    line = (f"  {e['epoch']:>3}  {e['tr_loss']:.4f}  {e['tr_auc']:.4f}  "
            f"{cm:>11.4f}  {hm:>11.4f}  {cm-hm:+.4f}")
    print(line)

In [ ]:
# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    backbone = args.get("backbone", BACKBONE)
    model = MultimodalClassifier(backbone=backbone, pretrained=False,
                                 dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    print(f"  Loaded ckpt epoch={state.get('epoch')}  backbone={backbone}  "
          f"train_auc_last={args.get('train_auc_last', 'n/a')}")
    return model

def predict(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass on test...")
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN}) "
      f"using last-epoch ckpt (final train_auc={history[-1]['tr_auc']:.4f})")

t0 = time.time()
preds = predict(ckpt_path, test_loader, tta_scales=scales_to_use)
print(f"  done in {time.time()-t0:.1f}s")

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.4f}, "
      f"min {preds.min():.4f}, max {preds.max():.4f})")
print(f"  <0.05: {(preds < 0.05).mean():.2%}    >0.95: {(preds > 0.95).mean():.2%}")
print(sub.head())
!wc -l /kaggle/working/submission.csv